#API Data Pulling 
##Claude Assisted Coding

In [ ]:
##Pull APPL 13F filings from SEC.gov, filter to Apple holdings, and save to CSV for SQL stage.
#Imports and config
import requests
import zipfile
import io
import pandas as pd
from pathlib import Path
import time

AAPL_CUSIP = "037833100"  

HEADERS = {"User-Agent": "Enoch <enochzhang56@gmail.com> Personal Research Project"}

RAW_DIR = Path("data/raw_13f")
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2018
START_Q = 1
END_YEAR = 2026     
END_Q = 1           

def quarter_range(start_year, start_q, end_year, end_q):
    y, q = start_year, start_q
    while (y, q) <= (end_year, end_q):
        yield y, q
        q += 1
        if q > 4:
            q = 1
            y += 1

QUARTERS = list(quarter_range(START_YEAR, START_Q, END_YEAR, END_Q))
print(f"Will pull {len(QUARTERS)} quarters: {QUARTERS[0]} through {QUARTERS[-1]}")

def download_quarter_zip(year: int, q: int) -> Path:
    fname = f"{year}q{q}_form13f.zip"
    local_path = RAW_DIR / fname
    if local_path.exists():
        return local_path  # already cached, skip re-download

    url = f"https://www.sec.gov/files/structureddata/data/form-13f-data-sets/{fname}"
    resp = requests.get(url, headers=HEADERS, timeout=60)
    if resp.status_code != 200:
        print(f"  [WARN] {fname} -> HTTP {resp.status_code}, skipping")
        return None

    local_path.write_bytes(resp.content)
    time.sleep(0.5)  # be polite to SEC's servers
    return local_path

# %% Parse function
def parse_quarter_zip(zip_path: Path) -> pd.DataFrame:
    if zip_path is None:
        return pd.DataFrame()

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()

        def read_tsv(target_name):
            match = [n for n in names if n.upper().endswith(target_name)]
            if not match:
                return pd.DataFrame()
            with z.open(match[0]) as f:
                return pd.read_csv(f, sep="\t", encoding="latin-1", low_memory=False)

        submission = read_tsv("SUBMISSION.TSV")
        coverpage = read_tsv("COVERPAGE.TSV")
        infotable = read_tsv("INFOTABLE.TSV")

    if infotable.empty:
        return pd.DataFrame()

    aapl = infotable[infotable["CUSIP"] == AAPL_CUSIP].copy()
    if aapl.empty:
        return pd.DataFrame()

    aapl = aapl.merge(
        submission[["ACCESSION_NUMBER", "CIK", "PERIODOFREPORT", "SUBMISSIONTYPE"]],
        on="ACCESSION_NUMBER", how="left"
    )
    aapl = aapl.merge(
        coverpage[["ACCESSION_NUMBER", "FILINGMANAGER_NAME"]],
        on="ACCESSION_NUMBER", how="left"
    )

    return aapl

all_quarters = []

for year, q in QUARTERS:
    print(f"Processing {year}Q{q}...")
    zpath = download_quarter_zip(year, q)
    df_q = parse_quarter_zip(zpath)
    if not df_q.empty:
        df_q["source_year"] = year
        df_q["source_quarter"] = q
        all_quarters.append(df_q)
    print(f"  -> {len(df_q)} AAPL holding rows")

aapl_master = pd.concat(all_quarters, ignore_index=True)
print(f"\nTotal AAPL holding-rows across all quarters: {len(aapl_master)}")

print(aapl_master[["FILINGMANAGER_NAME", "PERIODOFREPORT", "VALUE", "SSHPRNAMT"]].head(10))
print("\nUnique managers:", aapl_master["CIK"].nunique())
print("Quarters covered:", sorted(aapl_master["PERIODOFREPORT"].unique()))


out_path = OUT_DIR / "aapl_13f_holdings_2018_present.csv"
aapl_master.to_csv(out_path, index=False)
print(f"Saved: {out_path}")






Will pull 33 quarters: (2018, 1) through (2026, 1)
Processing 2018Q1...
  -> 4579 AAPL holding rows
Processing 2018Q2...
  -> 4751 AAPL holding rows
Processing 2018Q3...
  -> 4955 AAPL holding rows
Processing 2018Q4...
  -> 5101 AAPL holding rows
Processing 2019Q1...
  -> 5252 AAPL holding rows
Processing 2019Q2...
  -> 5786 AAPL holding rows
Processing 2019Q3...
  -> 5925 AAPL holding rows
Processing 2019Q4...
  -> 5960 AAPL holding rows
Processing 2020Q1...
  -> 6429 AAPL holding rows
Processing 2020Q2...
  -> 7053 AAPL holding rows
Processing 2020Q3...
  -> 5815 AAPL holding rows
Processing 2020Q4...
  -> 6381 AAPL holding rows
Processing 2021Q1...
  -> 8483 AAPL holding rows
Processing 2021Q2...
  -> 6526 AAPL holding rows
Processing 2021Q3...
  -> 7110 AAPL holding rows
Processing 2021Q4...
  -> 6373 AAPL holding rows
Processing 2022Q1...
  -> 6772 AAPL holding rows
Processing 2022Q2...
  -> 6990 AAPL holding rows
Processing 2022Q3...
  -> 7054 AAPL holding rows
Processing 2022Q4.

In [ ]:
##API Pull did not collect 2023 foward. Readjusted code to collect 2024-2026 Data. 

AAPL_CUSIP = "037833100"  # Apple Inc, common stock

HEADERS = {"User-Agent": "Enoch <enochzhang56@gmail.com> Personal Research Project"}

RAW_DIR = Path("data/raw_13f")
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://www.sec.gov/files/structureddata/data/form-13f-data-sets"

OLD_STYLE_FILES = [
    f"{year}q{q}_form13f.zip"
    for year in range(2018, 2024)
    for q in range(1, 5)
]
NEW_STYLE_FILES = [
    "01jan2024-29feb2024_form13f.zip",
    "01mar2024-31may2024_form13f.zip",
    "01jun2024-31aug2024_form13f.zip",
    "01sep2024-30nov2024_form13f.zip",
    "01dec2024-28feb2025_form13f.zip",
    "01mar2025-31may2025_form13f.zip",
    "01jun2025-31aug2025_form13f.zip",
    "01sep2025-30nov2025_form13f.zip",
    "01dec2025-28feb2026_form13f.zip",
    "01mar2026-31may2026_form13f.zip",
]

ALL_FILES = OLD_STYLE_FILES + NEW_STYLE_FILES
print(f"Will attempt {len(ALL_FILES)} files: {ALL_FILES[0]} through {ALL_FILES[-1]}")

def download_zip(fname: str) -> Path:
    local_path = RAW_DIR / fname
    if local_path.exists():
        return local_path  

    url = f"{BASE_URL}/{fname}"
    resp = requests.get(url, headers=HEADERS, timeout=60)
    if resp.status_code != 200:
        print(f"  [WARN] {fname} -> HTTP {resp.status_code}, skipping")
        return None

    local_path.write_bytes(resp.content)
    time.sleep(0.5)  
    return local_path

def parse_zip(zip_path: Path) -> pd.DataFrame:    
    if zip_path is None:
        return pd.DataFrame()

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()

        def read_tsv(target_name):
            match = [n for n in names if n.upper().endswith(target_name)]
            if not match:
                return pd.DataFrame()
            with z.open(match[0]) as f:
                return pd.read_csv(f, sep="\t", encoding="latin-1", low_memory=False)

        submission = read_tsv("SUBMISSION.TSV")
        coverpage = read_tsv("COVERPAGE.TSV")
        infotable = read_tsv("INFOTABLE.TSV")

    if infotable.empty:
        return pd.DataFrame()

    aapl = infotable[infotable["CUSIP"] == AAPL_CUSIP].copy()
    if aapl.empty:
        return pd.DataFrame()

    aapl = aapl.merge(
        submission[["ACCESSION_NUMBER", "CIK", "PERIODOFREPORT", "SUBMISSIONTYPE"]],
        on="ACCESSION_NUMBER", how="left"
    )
    aapl = aapl.merge(
        coverpage[["ACCESSION_NUMBER", "FILINGMANAGER_NAME"]],
        on="ACCESSION_NUMBER", how="left"
    )

    return aapl

all_batches = []

for fname in ALL_FILES:
    print(f"Processing {fname}...")
    zpath = download_zip(fname)
    df_batch = parse_zip(zpath)
    if not df_batch.empty:
        all_batches.append(df_batch)
    print(f"  -> {len(df_batch)} AAPL holding rows")

aapl_master = pd.concat(all_batches, ignore_index=True)

aapl_master = aapl_master.sort_values("ACCESSION_NUMBER").drop_duplicates(
    subset=["CIK", "PERIODOFREPORT"], keep="last"
)

print(f"\nTotal AAPL holding-rows after de-dupe: {len(aapl_master)}")

print(aapl_master[["FILINGMANAGER_NAME", "PERIODOFREPORT", "VALUE", "SSHPRNAMT"]].head(10))
print("\nUnique managers:", aapl_master["CIK"].nunique())
print("Quarters covered:", sorted(aapl_master["PERIODOFREPORT"].unique()))

out_path = OUT_DIR / "aapl_13f_holdings_2018_present.csv"
aapl_master.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Will attempt 34 files: 2018q1_form13f.zip through 01mar2026-31may2026_form13f.zip
Processing 2018q1_form13f.zip...
  -> 4579 AAPL holding rows
Processing 2018q2_form13f.zip...
  -> 4751 AAPL holding rows
Processing 2018q3_form13f.zip...
  -> 4955 AAPL holding rows
Processing 2018q4_form13f.zip...
  -> 5101 AAPL holding rows
Processing 2019q1_form13f.zip...
  -> 5252 AAPL holding rows
Processing 2019q2_form13f.zip...
  -> 5786 AAPL holding rows
Processing 2019q3_form13f.zip...
  -> 5925 AAPL holding rows
Processing 2019q4_form13f.zip...
  -> 5960 AAPL holding rows
Processing 2020q1_form13f.zip...
  -> 6429 AAPL holding rows
Processing 2020q2_form13f.zip...
  -> 7053 AAPL holding rows
Processing 2020q3_form13f.zip...
  -> 5815 AAPL holding rows
Processing 2020q4_form13f.zip...
  -> 6381 AAPL holding rows
Processing 2021q1_form13f.zip...
  -> 8483 AAPL holding rows
Processing 2021q2_form13f.zip...
  -> 6526 AAPL holding rows
Processing 2021q3_form13f.zip...
  -> 7110 AAPL holding rows
Pro

In [ ]:

#Pull macro (FRED), price/volatility (yfinance), and earnings/valuation (Alpha Vantage) data, aggregated to quarter-end.
# Primary Key PERIODOFREPORT.

import pandas as pd
import numpy as np
import requests
import yfinance as yf
from pathlib import Path
import time

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FRED_API_KEY = "hidden"
ALPHA_VANTAGE_KEY = "hidden"

START_DATE = "2018-01-01"
END_DATE = "2026-06-30"

MEGACAP_PEERS = ["MSFT", "NVDA", "META", "GOOGL"]

def fetch_fred_series(series_id: str) -> pd.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": START_DATE,
        "observation_end": END_DATE,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    obs = resp.json()["observations"]
    df = pd.DataFrame(obs)[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")  
    df = df.rename(columns={"value": series_id})
    return df.set_index("date")

FRED_SERIES = {
    "FEDFUNDS": "fed_funds_rate",
    "DGS10": "yield_10y",
    "DGS2": "yield_2y",
    "CPIAUCSL": "cpi_index",
    "VIXCLS": "vix",
}

fred_frames = []
for series_id in FRED_SERIES:
    print(f"Pulling FRED series: {series_id}")
    fred_frames.append(fetch_fred_series(series_id))
    time.sleep(0.5)

fred_daily = pd.concat(fred_frames, axis=1).rename(columns=FRED_SERIES)
fred_daily = fred_daily.sort_index()

fred_quarterly = fred_daily.resample("QE").agg({
    "fed_funds_rate": "last",   
    "yield_10y": "last",
    "yield_2y": "last",
    "cpi_index": "last",
    "vix": "mean",             
})

fred_quarterly["yield_spread_10y_2y"] = (
    fred_quarterly["yield_10y"] - fred_quarterly["yield_2y"]
)
fred_quarterly["cpi_yoy_pct"] = fred_quarterly["cpi_index"].pct_change(4) * 100
fred_quarterly["fed_funds_rate_change_qoq"] = fred_quarterly["fed_funds_rate"].diff()

fred_quarterly = fred_quarterly.reset_index().rename(columns={"date": "quarter_end"})
print(fred_quarterly.tail())


def quarterly_return_and_vol(ticker: str) -> pd.DataFrame:
    hist = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)

    if isinstance(hist.columns, pd.MultiIndex):
        close = hist["Close"][ticker]
    else:
        close = hist["Close"]

    daily_ret = close.pct_change()
    q_close = close.resample("QE").last()
    q_return = q_close.pct_change()
    q_vol = daily_ret.resample("QE").std() * np.sqrt(63)  

    out = pd.DataFrame({
        "quarter_end": q_return.index,
        f"{ticker}_return": q_return.values,
        f"{ticker}_volatility": q_vol.values,
    })
    return out

print("Pulling AAPL price data...")
aapl_price = quarterly_return_and_vol("AAPL")
aapl_price = aapl_price.rename(columns={
    "AAPL_return": "aapl_price_return",
    "AAPL_volatility": "aapl_realized_volatility",
})

print("Pulling peer price data for sector-rotation control...")
peer_frames = []
for ticker in MEGACAP_PEERS:
    peer_frames.append(quarterly_return_and_vol(ticker).set_index("quarter_end"))
    time.sleep(0.5)

peer_returns = pd.concat(peer_frames, axis=1)
return_cols = [c for c in peer_returns.columns if c.endswith("_return")]
peer_returns["megacap_peer_avg_return"] = peer_returns[return_cols].mean(axis=1)
peer_returns = peer_returns[["megacap_peer_avg_return"]].reset_index()

def fetch_alpha_vantage_earnings(symbol: str) -> pd.DataFrame:
    url = "https://www.alphavantage.co/query"
    params = {"function": "EARNINGS", "symbol": symbol, "apikey": ALPHA_VANTAGE_KEY}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    q = pd.DataFrame(data["quarterlyEarnings"])
    q["fiscalDateEnding"] = pd.to_datetime(q["fiscalDateEnding"])
    for col in ["reportedEPS", "estimatedEPS", "surprisePercentage"]:
        q[col] = pd.to_numeric(q[col], errors="coerce")
    return q[["fiscalDateEnding", "reportedEPS", "estimatedEPS", "surprisePercentage"]]

print("Pulling AAPL earnings from Alpha Vantage...")
aapl_earnings = fetch_alpha_vantage_earnings("AAPL")
aapl_earnings = aapl_earnings.rename(columns={
    "fiscalDateEnding": "quarter_end",
    "reportedEPS": "eps_actual",
    "estimatedEPS": "eps_estimate",
    "surprisePercentage": "earnings_surprise_pct",
})

aapl_earnings.to_csv(OUT_DIR / "aapl_earnings_raw.csv", index=False)

market_macro = (
    fred_quarterly
    .merge(aapl_price, on="quarter_end", how="left")
    .merge(peer_returns, on="quarter_end", how="left")
    .merge(aapl_earnings, on="quarter_end", how="left")
)

print(market_macro.tail(10))

out_path = OUT_DIR / "quarterly_macro_market.csv"
market_macro.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Pulling FRED series: FEDFUNDS
Pulling FRED series: DGS10
Pulling FRED series: DGS2
Pulling FRED series: CPIAUCSL
Pulling FRED series: VIXCLS


/var/folders/4g/7t5q5qys145_ll4nj4bl8dwh0000gn/T/ipykernel_55522/1453027573.py:57: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  fred_daily = pd.concat(fred_frames, axis=1).rename(columns=FRED_SERIES)


   quarter_end  fed_funds_rate  yield_10y  yield_2y  cpi_index        vix  \
29  2025-06-30            4.33       4.24      3.72    321.435  23.561406   
30  2025-09-30            4.22       4.16      3.60    324.245  15.983030   
31  2025-12-31            3.72       4.18      3.47    326.031  17.745231   
32  2026-03-31            3.64       4.30      3.79    330.293  20.429048   
33  2026-06-30            3.63       4.44      4.14    332.568  18.312188   

    yield_spread_10y_2y  cpi_yoy_pct  fed_funds_rate_change_qoq  
29                 0.52     2.680454                       0.00  
30                 0.56     3.022572                      -0.11  
31                 0.71     2.653304                      -0.50  
32                 0.51     3.285958                      -0.08  
33                 0.30     3.463531                      -0.01  
Pulling AAPL price data...
Pulling peer price data for sector-rotation control...
Pulling AAPL earnings from Alpha Vantage...
   quarter_end 

In [ ]:
#Reformat 13F summary pages to get manager-level AUM, then merge onto AAPL holdings for % of manager AUM.

import zipfile
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw_13f")
OUT_DIR = Path("data/processed")

OLD_STYLE_FILES = [
    f"{year}q{q}_form13f.zip"
    for year in range(2018, 2024)
    for q in range(1, 5)
]
NEW_STYLE_FILES = [
    "01jan2024-29feb2024_form13f.zip",
    "01mar2024-31may2024_form13f.zip",
    "01jun2024-31aug2024_form13f.zip",
    "01sep2024-30nov2024_form13f.zip",
    "01dec2024-28feb2025_form13f.zip",
    "01mar2025-31may2025_form13f.zip",
    "01jun2025-31aug2025_form13f.zip",
    "01sep2025-30nov2025_form13f.zip",
    "01dec2025-28feb2026_form13f.zip",
    "01mar2026-31may2026_form13f.zip",
]
ALL_FILES = OLD_STYLE_FILES + NEW_STYLE_FILES

def extract_summarypage(zip_path: Path) -> pd.DataFrame:
    if not zip_path.exists():
        return pd.DataFrame()
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        match = [n for n in names if n.upper().endswith("SUMMARYPAGE.TSV")]
        if not match:
            return pd.DataFrame()
        with z.open(match[0]) as f:
            df = pd.read_csv(f, sep="\t", encoding="latin-1", low_memory=False)
    return df[["ACCESSION_NUMBER", "TABLEVALUETOTAL"]]

summary_frames = []
for fname in ALL_FILES:
    zpath = RAW_DIR / fname
    df_s = extract_summarypage(zpath)
    if not df_s.empty:
        summary_frames.append(df_s)
    print(f"{fname}: {len(df_s)} rows")

manager_aum = pd.concat(summary_frames, ignore_index=True)
manager_aum = manager_aum.drop_duplicates(subset="ACCESSION_NUMBER", keep="last")
manager_aum = manager_aum.rename(columns={"TABLEVALUETOTAL": "manager_total_aum"})

holdings_path = OUT_DIR / "aapl_13f_holdings_2018_present.csv"
holdings = pd.read_csv(holdings_path)

holdings = holdings.merge(manager_aum, on="ACCESSION_NUMBER", how="left")


holdings["pct_of_manager_aum"] = holdings["VALUE"] / holdings["manager_total_aum"]

missing_aum = holdings["manager_total_aum"].isna().sum()
print(f"Rows missing manager_total_aum: {missing_aum} of {len(holdings)}")

holdings.to_csv(holdings_path, index=False)
print(f"Updated: {holdings_path}")

2018q1_form13f.zip: 5017 rows
2018q2_form13f.zip: 5081 rows
2018q3_form13f.zip: 5087 rows
2018q4_form13f.zip: 5112 rows
2019q1_form13f.zip: 5328 rows
2019q2_form13f.zip: 5534 rows
2019q3_form13f.zip: 5381 rows
2019q4_form13f.zip: 5359 rows
2020q1_form13f.zip: 5727 rows
2020q2_form13f.zip: 5757 rows
2020q3_form13f.zip: 5681 rows
2020q4_form13f.zip: 5771 rows
2021q1_form13f.zip: 6375 rows
2021q2_form13f.zip: 6390 rows
2021q3_form13f.zip: 6501 rows
2021q4_form13f.zip: 6430 rows
2022q1_form13f.zip: 7251 rows
2022q2_form13f.zip: 7292 rows
2022q3_form13f.zip: 7321 rows
2022q4_form13f.zip: 7308 rows
2023q1_form13f.zip: 7627 rows
2023q2_form13f.zip: 7513 rows
2023q3_form13f.zip: 7522 rows
2023q4_form13f.zip: 7624 rows
01jan2024-29feb2024_form13f.zip: 7864 rows
01mar2024-31may2024_form13f.zip: 8241 rows
01jun2024-31aug2024_form13f.zip: 8244 rows
01sep2024-30nov2024_form13f.zip: 8249 rows
01dec2024-28feb2025_form13f.zip: 8848 rows
01mar2025-31may2025_form13f.zip: 9032 rows
01jun2025-31aug2025_fo

In [ ]:
#Reformat 13F summary pages to get manager-level AUM, then merge onto AAPL holdings for % of manager AUM.

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FRED_API_KEY = "hidden"
ALPHA_VANTAGE_KEY = "hidden"

START_DATE = "2018-01-01"
END_DATE = "2026-06-30"

MEGACAP_PEERS = ["MSFT", "NVDA", "META", "GOOGL"]
BROAD_MARKET_TICKER = "SPY"  


def fetch_fred_series(series_id: str) -> pd.DataFrame:
    """Pull a daily/monthly FRED series as a DataFrame of (date, value)."""
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": START_DATE,
        "observation_end": END_DATE,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    obs = resp.json()["observations"]
    df = pd.DataFrame(obs)[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")  
    df = df.rename(columns={"value": series_id})
    return df.set_index("date")

FRED_SERIES = {
    "FEDFUNDS": "fed_funds_rate",
    "DGS10": "yield_10y",
    "DGS2": "yield_2y",
    "CPIAUCSL": "cpi_index",
    "VIXCLS": "vix",
}

fred_frames = []
for series_id in FRED_SERIES:
    print(f"Pulling FRED series: {series_id}")
    fred_frames.append(fetch_fred_series(series_id))
    time.sleep(0.5)

fred_daily = pd.concat(fred_frames, axis=1).rename(columns=FRED_SERIES)
fred_daily = fred_daily.sort_index()

fred_quarterly = fred_daily.resample("QE").agg({
    "fed_funds_rate": "last",   
    "yield_10y": "last",
    "yield_2y": "last",
    "cpi_index": "last",
    "vix": "mean",            
})

fred_quarterly["yield_spread_10y_2y"] = (
    fred_quarterly["yield_10y"] - fred_quarterly["yield_2y"]
)
fred_quarterly["cpi_yoy_pct"] = fred_quarterly["cpi_index"].pct_change(4) * 100
fred_quarterly["fed_funds_rate_change_qoq"] = fred_quarterly["fed_funds_rate"].diff()

fred_quarterly = fred_quarterly.reset_index().rename(columns={"date": "quarter_end"})
print(fred_quarterly.tail())

def quarterly_return_and_vol(ticker: str) -> pd.DataFrame:
    """Daily prices -> quarterly return and realized volatility."""
    hist = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)

    if isinstance(hist.columns, pd.MultiIndex):
        close = hist["Close"][ticker]
    else:
        close = hist["Close"]

    daily_ret = close.pct_change()

    q_close = close.resample("QE").last()
    q_return = q_close.pct_change()
    q_vol = daily_ret.resample("QE").std() * np.sqrt(63)  

    out = pd.DataFrame({
        "quarter_end": q_return.index,
        f"{ticker}_return": q_return.values,
        f"{ticker}_volatility": q_vol.values,
        f"{ticker}_quarter_close": q_close.values,  
    })
    return out

print("Pulling AAPL price data...")
aapl_price = quarterly_return_and_vol("AAPL")
aapl_price = aapl_price.rename(columns={
    "AAPL_return": "aapl_price_return",
    "AAPL_volatility": "aapl_realized_volatility",
    "AAPL_quarter_close": "aapl_quarter_close",
})

print("Pulling SPY (broad market) price data...")
spy_price = quarterly_return_and_vol(BROAD_MARKET_TICKER)
spy_price = spy_price.rename(columns={
    "SPY_return": "spy_return",
    "SPY_volatility": "spy_volatility",
})[["quarter_end", "spy_return", "spy_volatility"]]

print("Pulling peer price data for sector-rotation control...")
peer_frames = []
for ticker in MEGACAP_PEERS:
    peer_frames.append(quarterly_return_and_vol(ticker).set_index("quarter_end"))
    time.sleep(0.5)

peer_returns = pd.concat(peer_frames, axis=1)
return_cols = [c for c in peer_returns.columns if c.endswith("_return")]
peer_returns["megacap_peer_avg_return"] = peer_returns[return_cols].mean(axis=1)
peer_returns = peer_returns[["megacap_peer_avg_return"]].reset_index()

def fetch_alpha_vantage_earnings(symbol: str) -> pd.DataFrame:
    url = "https://www.alphavantage.co/query"
    params = {"function": "EARNINGS", "symbol": symbol, "apikey": ALPHA_VANTAGE_KEY}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    q = pd.DataFrame(data["quarterlyEarnings"])
    q["fiscalDateEnding"] = pd.to_datetime(q["fiscalDateEnding"])
    for col in ["reportedEPS", "estimatedEPS", "surprisePercentage"]:
        q[col] = pd.to_numeric(q[col], errors="coerce")
    return q[["fiscalDateEnding", "reportedEPS", "estimatedEPS", "surprisePercentage"]]

print("Pulling AAPL earnings from Alpha Vantage...")
aapl_earnings = fetch_alpha_vantage_earnings("AAPL")
aapl_earnings = aapl_earnings.rename(columns={
    "fiscalDateEnding": "quarter_end",
    "reportedEPS": "eps_actual",
    "estimatedEPS": "eps_estimate",
    "surprisePercentage": "earnings_surprise_pct",
})

aapl_earnings.to_csv(OUT_DIR / "aapl_earnings_raw.csv", index=False)

market_macro = (
    fred_quarterly
    .merge(aapl_price, on="quarter_end", how="left")
    .merge(spy_price, on="quarter_end", how="left")
    .merge(peer_returns, on="quarter_end", how="left")
    .merge(aapl_earnings, on="quarter_end", how="left")
)

market_macro = market_macro.sort_values("quarter_end").reset_index(drop=True)
market_macro["trailing_4q_eps"] = market_macro["eps_actual"].rolling(4, min_periods=4).sum()
market_macro["trailing_pe"] = market_macro["aapl_quarter_close"] / market_macro["trailing_4q_eps"]

print(market_macro.tail(10))

out_path = OUT_DIR / "quarterly_macro_market.csv"
market_macro.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Pulling FRED series: FEDFUNDS
Pulling FRED series: DGS10
Pulling FRED series: DGS2
Pulling FRED series: CPIAUCSL
Pulling FRED series: VIXCLS


/var/folders/4g/7t5q5qys145_ll4nj4bl8dwh0000gn/T/ipykernel_55522/1318977446.py:60: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  fred_daily = pd.concat(fred_frames, axis=1).rename(columns=FRED_SERIES)


   quarter_end  fed_funds_rate  yield_10y  yield_2y  cpi_index        vix  \
29  2025-06-30            4.33       4.24      3.72    321.435  23.561406   
30  2025-09-30            4.22       4.16      3.60    324.245  15.983030   
31  2025-12-31            3.72       4.18      3.47    326.031  17.745231   
32  2026-03-31            3.64       4.30      3.79    330.293  20.429048   
33  2026-06-30            3.63       4.44      4.14    332.568  18.312188   

    yield_spread_10y_2y  cpi_yoy_pct  fed_funds_rate_change_qoq  
29                 0.52     2.680454                       0.00  
30                 0.56     3.022572                      -0.11  
31                 0.71     2.653304                      -0.50  
32                 0.51     3.285958                      -0.08  
33                 0.30     3.463531                      -0.01  
Pulling AAPL price data...
Pulling SPY (broad market) price data...
Pulling peer price data for sector-rotation control...
Pulling AAPL earni

In [ ]:

# Reprocess cached zips for each AAPL-holding manager's FULL portfolio 

RAW_DIR = Path("data/raw_13f")
OUT_DIR = Path("data/processed")

OLD_STYLE_FILES = [
    f"{year}q{q}_form13f.zip"
    for year in range(2018, 2024)
    for q in range(1, 5)
]
NEW_STYLE_FILES = [
    "01jan2024-29feb2024_form13f.zip",
    "01mar2024-31may2024_form13f.zip",
    "01jun2024-31aug2024_form13f.zip",
    "01sep2024-30nov2024_form13f.zip",
    "01dec2024-28feb2025_form13f.zip",
    "01mar2025-31may2025_form13f.zip",
    "01jun2025-31aug2025_form13f.zip",
    "01sep2025-30nov2025_form13f.zip",
    "01dec2025-28feb2026_form13f.zip",
    "01mar2026-31may2026_form13f.zip",
]
ALL_FILES = OLD_STYLE_FILES + NEW_STYLE_FILES

INDEX_FUND_NAME_MATCHES = [
    "VANGUARD", "BLACKROCK", "STATE STREET", "SSGA",
    "FIDELITY", "SCHWAB", "GEODE", "NORTHERN TRUST",
]

holdings = pd.read_csv(OUT_DIR / "aapl_13f_holdings_2018_present.csv")
target_accessions = set(holdings["ACCESSION_NUMBER"].unique())
print(f"Targeting {len(target_accessions)} filings (manager x quarter)")

def parse_full_portfolio(zip_path: Path) -> pd.DataFrame:
    if not zip_path.exists():
        return pd.DataFrame()

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()

        def read_tsv(target_name):
            match = [n for n in names if n.upper().endswith(target_name)]
            if not match:
                return pd.DataFrame()
            with z.open(match[0]) as f:
                return pd.read_csv(f, sep="\t", encoding="latin-1", low_memory=False)

        submission = read_tsv("SUBMISSION.TSV")
        coverpage = read_tsv("COVERPAGE.TSV")
        infotable = read_tsv("INFOTABLE.TSV")

    if infotable.empty:
        return pd.DataFrame()

    infotable = infotable[infotable["ACCESSION_NUMBER"].isin(target_accessions)]
    if infotable.empty:
        return pd.DataFrame()

    infotable = infotable.merge(
        submission[["ACCESSION_NUMBER", "CIK", "PERIODOFREPORT"]],
        on="ACCESSION_NUMBER", how="left"
    )
    infotable = infotable.merge(
        coverpage[["ACCESSION_NUMBER", "FILINGMANAGER_NAME"]],
        on="ACCESSION_NUMBER", how="left"
    )

    return infotable[[
        "ACCESSION_NUMBER", "CIK", "FILINGMANAGER_NAME", "PERIODOFREPORT",
        "CUSIP", "NAMEOFISSUER", "VALUE", "SSHPRNAMT",
    ]]

all_positions = []
for fname in ALL_FILES:
    zpath = RAW_DIR / fname
    df_p = parse_full_portfolio(zpath)
    if not df_p.empty:
        all_positions.append(df_p)
    print(f"{fname}: {len(df_p)} rows")

full_portfolio = pd.concat(all_positions, ignore_index=True)

full_portfolio = full_portfolio.sort_values("ACCESSION_NUMBER").drop_duplicates(
    subset=["ACCESSION_NUMBER", "CUSIP"], keep="last"
)

print(f"\nTotal position-rows across all target filings: {len(full_portfolio)}")

full_portfolio.to_csv(OUT_DIR / "manager_full_portfolio.csv", index=False)

manager_summary = full_portfolio.groupby(
    ["ACCESSION_NUMBER", "CIK", "FILINGMANAGER_NAME", "PERIODOFREPORT"], as_index=False
).agg(
    manager_position_count=("CUSIP", "count"),
    manager_total_value_recomputed=("VALUE", "sum"),
)

name_pattern = "|".join(INDEX_FUND_NAME_MATCHES)
manager_summary["is_index_fund_flag"] = (
    manager_summary["FILINGMANAGER_NAME"]
    .str.upper()
    .str.contains(name_pattern, na=False)
    .astype(int)
)

print(manager_summary.head(10))
print(f"\nFlagged as index funds: {manager_summary['is_index_fund_flag'].sum()} of {len(manager_summary)} filings")

manager_summary.to_csv(OUT_DIR / "manager_quarter_summary.csv", index=False)
print("Saved: manager_full_portfolio.csv and manager_quarter_summary.csv")

Targeting 144370 filings (manager x quarter)
2018q1_form13f.zip: 1616151 rows
2018q2_form13f.zip: 1632661 rows
2018q3_form13f.zip: 1620036 rows
2018q4_form13f.zip: 1702517 rows
2019q1_form13f.zip: 1716646 rows
2019q2_form13f.zip: 2030600 rows
2019q3_form13f.zip: 1857046 rows
2019q4_form13f.zip: 1843230 rows
2020q1_form13f.zip: 1910620 rows
2020q2_form13f.zip: 1913175 rows
2020q3_form13f.zip: 1862771 rows
2020q4_form13f.zip: 1938117 rows
2021q1_form13f.zip: 2089868 rows
2021q2_form13f.zip: 2067409 rows
2021q3_form13f.zip: 2311491 rows
2021q4_form13f.zip: 2222539 rows
2022q1_form13f.zip: 2300966 rows
2022q2_form13f.zip: 2276540 rows
2022q3_form13f.zip: 2252741 rows
2022q4_form13f.zip: 2319798 rows
2023q1_form13f.zip: 2359807 rows
2023q2_form13f.zip: 2243942 rows
2023q3_form13f.zip: 2350746 rows
2023q4_form13f.zip: 2541728 rows
01jan2024-29feb2024_form13f.zip: 2392629 rows
01mar2024-31may2024_form13f.zip: 2585139 rows
01jun2024-31aug2024_form13f.zip: 2857047 rows
01sep2024-30nov2024_form1

In [ ]:
#Stage all processed data to SQL database for joining

from sqlalchemy import create_engine

DB_USER = "postgres"
DB_PASSWORD = "hidden"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "postgres"


engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

DATA_DIR = "data/processed"

tables_to_load = {
    "raw_aapl_holdings": "aapl_13f_holdings_2018_present.csv",
    "raw_manager_full_portfolio": "manager_full_portfolio.csv",
    "raw_manager_quarter_summary": "manager_quarter_summary.csv",
    "raw_quarterly_macro_market": "quarterly_macro_market.csv",
}

for table_name, fname in tables_to_load.items():
    print(f"Loading {fname} -> {table_name}...")
    df = pd.read_csv(f"{DATA_DIR}/{fname}")
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=5000)
    print(f"  -> {len(df)} rows loaded")

print("\nAll tables loaded.")

with engine.connect() as conn:
    for table_name in tables_to_load:
        result = conn.exec_driver_sql(f"SELECT COUNT(*) FROM {table_name}")
        print(f"{table_name}: {result.scalar()} rows")



Loading aapl_13f_holdings_2018_present.csv -> raw_aapl_holdings...
  -> 144370 rows loaded
Loading manager_full_portfolio.csv -> raw_manager_full_portfolio...
  -> 53202619 rows loaded
Loading manager_quarter_summary.csv -> raw_manager_quarter_summary...
  -> 144370 rows loaded
Loading quarterly_macro_market.csv -> raw_quarterly_macro_market...
  -> 34 rows loaded

All tables loaded.
raw_aapl_holdings: 144370 rows
raw_manager_full_portfolio: 53202619 rows
raw_manager_quarter_summary: 144370 rows
raw_quarterly_macro_market: 34 rows


In [ ]:
#Fix APPL close price for unit validcation and restage appl_adjusted_close.csv to SQL

OUT_DIR = Path("data/processed")
START_DATE = "2018-01-01"
END_DATE = "2026-06-30"

DB_USER = "postgres"
DB_PASSWORD = "hidden"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "postgres"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

hist = yf.download("AAPL", start=START_DATE, end=END_DATE, progress=False, auto_adjust=False)

if isinstance(hist.columns, pd.MultiIndex):
    close = hist["Close"]["AAPL"]
else:
    close = hist["Close"]

q_close_unadjusted = close.resample("QE").last()

unadjusted_df = pd.DataFrame({
    "quarter_end": q_close_unadjusted.index,
    "aapl_quarter_close_unadjusted": q_close_unadjusted.values,
})
unadjusted_df.columns = unadjusted_df.columns.str.lower()

print(unadjusted_df.tail(10))

unadjusted_df.to_csv(OUT_DIR / "aapl_unadjusted_close.csv", index=False)
unadjusted_df.to_sql("aapl_unadjusted_close", engine, if_exists="replace", index=False)
print("Loaded aapl_unadjusted_close table.")

   quarter_end  aapl_quarter_close_unadjusted
24  2024-03-31                     171.479996
25  2024-06-30                     210.619995
26  2024-09-30                     233.000000
27  2024-12-31                     250.419998
28  2025-03-31                     222.130005
29  2025-06-30                     205.169998
30  2025-09-30                     254.630005
31  2025-12-31                     271.859985
32  2026-03-31                     253.789993
33  2026-06-30                     281.739990
Loaded aapl_unadjusted_close table.
